Aishwarya Srivastava

Group 6: Udita, Goerge, Evan

In [ ]:
#install libraries
!pip install transformers datasets peft accelerate bitsandbytes
import torch
import random
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import BitsAndBytesConfig, Trainer, TrainingArguments
from transformers import DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from datasets import Dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00


In [ ]:
!pip install -U bitsandbytes>=0.46.1
# Install this if the next cell is giving you an error.

In [ ]:
#1a

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# load the model for the later decision tree split
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
df = pd.read_csv("customer_support_dataset_200.csv")

#this will print the data frame that will showcase the decision split
print(df['department'].value_counts())

texts_train, texts_test, labels_train, labels_test = train_test_split(
    df['message'].tolist(),
    df['department'].tolist(),
    test_size=0.2,
    random_state=77
)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


department
Billing               50
Technical Support     50
Account Management    50
Sales                 50
Name: count, dtype: int64


In [ ]:
#1b

def classify_zero_shot(text):
    prompt = f"""Classify the following customer message into one of these categories:
                 Billing, Technical Support, Account Management, Sales.

Message: "{text}"
Department:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=20)
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return prediction.strip()

# there is need to run an example for each predition
print("Running zero-shot predictions on training set...")
predictions = [classify_zero_shot(t) for t in texts_train]

# now is the final step where it is normalizing predictions
def normalize(pred):
    pred = pred.lower()
    if "billing" in pred:
        return "Billing"
    if "technical" in pred or "support" in pred:
        return "Technical Support"
    if "account" in pred:
        return "Account Management"
    if "sales" in pred:
        return "Sales"
    return "Unknown"

norm_preds = [normalize(p) for p in predictions]
test_predictions = [classify_zero_shot(t) for t in texts_test]
norm_test_preds = [normalize(p) for p in test_predictions]



In [ ]:
from sklearn.metrics import accuracy_score
# this is the model accuracy on training set
train_accuracy = accuracy_score(labels_train, norm_preds)
print("Zero Shot Base Model Train Set accuracy:", train_accuracy*100, "%")

# this is the model accuracy on test set
test_accuracy = accuracy_score(labels_test, norm_test_preds)
print("Zero Shot Base Model accuracy on Test Set", test_accuracy*100, "%")



Zero Shot Base Model Train Set accuracy: 30.0 %
Zero Shot Base Model accuracy on Test Set 35.0 %


In [ ]:
#1c

# PART (c) LoRA Fine Tuning with Flan T5 Small

# Prepare the tokenized dataset

# Format inputs as instruction prompts and targets as plain department labels
input_texts  = [f"Classify this message: {t}" for t in texts_train]
target_texts = labels_train  # e.g. "Billing", "Technical Support", etc.

# Build a HuggingFace Dataset from the lists
dataset_dict = {
    "input_text":  input_texts,
    "target_text": target_texts,
}
dataset = Dataset.from_dict(dataset_dict)

# Tokenize both the input prompt and the target label
def tokenize_fn(batch):
    return tokenizer(
        batch["input_text"],
        text_target=batch["target_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["input_text", "target_text"],
)

print("✅ Dataset tokenized")
print(f"Training examples: {len(tokenized_dataset)}")
print(f"Features: {tokenized_dataset.features}")

#  LoRA configuration

# LoRA injects small trainable rank r matrices into the attention layers.
# Only those matrices are updated the original weights stay frozen.
# This keeps training fast and memory efficient on the free Colab GPU.
lora_config = LoraConfig(
    r=16,                      # rank of the low rank decomposition
    lora_alpha=32,             # scaling factor (effective lr = alpha/r)
    target_modules=["q", "v"],  # inject into query and value projections
    lora_dropout=0.05,         # small dropout for regularisation
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

tuned_model = get_peft_model(model, lora_config)

# Show how many parameters are actually being trained
tuned_model.print_trainable_parameters()
# Expected output: roughly 0.3 % of total params — everything else is frozen



# Data collator

# Seq2Seq collator handles dynamic padding of encoder + decoder sequences
# and shifts decoder input_ids by one position (teacher forcing).
data_collator = DataCollatorForSeq2Seq(tokenizer, model=tuned_model)


# Training arguments

training_args = TrainingArguments(
    output_dir="./lora_out",
    per_device_train_batch_size=8,   # 8 fits comfortably on a T4
    gradient_accumulation_steps=1,
    num_train_epochs=3,              # 2-3 epochs as specified
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,                       # mixed precision — faster on T4
    report_to="none",                # suppress W&B / other trackers
)


#Train

trainer = Trainer(
    model=tuned_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("🚀 Starting LoRA fine-tuning …")
trainer.train()
print("✅ Fine-tuning complete")


# Save the LoRA-adapted model

tuned_model.save_pretrained("./lora_flan_t5")
tokenizer.save_pretrained("./lora_flan_t5")
print("✅ Model saved to ./lora_flan_t5")


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

✅ Dataset tokenized
Training examples: 160
Features: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}
trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862
🚀 Starting LoRA fine-tuning …


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

#  Classify with the fine-tuned model

def classify_finetuned(text: str) -> str:
    """Run classification with the LoRA fine-tuned model."""
    # Use the same prompt format seen during training
    prompt = f"Classify this message: {text}"
    inputs  = tokenizer(prompt, return_tensors="pt").to(tuned_model.device)
    outputs = tuned_model.generate(**inputs, max_new_tokens=20)
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return prediction.strip()


#Run predictions on the TEST set

print("Running fine-tuned predictions on test set …")
raw_preds_ft  = [classify_finetuned(t) for t in texts_test]
norm_preds_ft = [normalize(p) for p in raw_preds_ft]   # reuse normalize() from Part b

# Compute accuracy

# Fine tuned accuracy
accuracy_ft = accuracy_score(labels_test, norm_preds_ft)

# Baseline accuracy (calculated in Part b)
# accuracy_zs was already computed — reuse it here
improvement = accuracy_ft - test_accuracy   # test_accuracy = zero-shot test accuracy from Part b

print("=" * 55)
print(f"  Zero-shot test accuracy   : {test_accuracy:.2%}")
print(f"  Fine-tuned test accuracy  : {accuracy_ft:.2%}")
print(f"  Improvement               : +{improvement:.2%}")
print("=" * 55)

#Detailed per class report

# Filter out any "Unknown" predictions for the sklearn report
# (Unknown means the model returned something outside the four classes)
valid_idx = [i for i, p in enumerate(norm_preds_ft) if p != "Unknown"]
if len(valid_idx) < len(norm_preds_ft):
    n_unknown = len(norm_preds_ft) - len(valid_idx)
    print(f"\n⚠️  {n_unknown} prediction(s) could not be mapped to a class label.")

labels_test_valid = [labels_test[i]   for i in valid_idx]
preds_ft_valid    = [norm_preds_ft[i] for i in valid_idx]

# Define all possible labels explicitly
all_possible_labels = ["Account Management", "Billing", "Sales", "Technical Support"]

print("\nClassification Report — Fine-Tuned Model (Test Set):")
print(classification_report(
    labels_test_valid,
    preds_ft_valid,
    labels=all_possible_labels,
    target_names=all_possible_labels,
    zero_division=0,
))

# Side by side sample comparison

print("\nSample predictions — first 5 test messages:")
print(f"{'Message':<45} {'True Label':<22} {'Zero-Shot':<22} {'Fine-Tuned'}")
print("-" * 115)
for i in range(min(5, len(texts_test))):
    msg      = texts_test[i][:42] + "…" if len(texts_test[i]) > 42 else texts_test[i]
    true_lbl = labels_test[i]
    zs_lbl   = norm_test_preds[i]    # from Part b
    ft_lbl   = norm_preds_ft[i]
    print(f"{msg:<45} {true_lbl:<22} {zs_lbl:<22} {ft_lbl}")

The accuracy improved here becuase the fine tuned model specializes a general purpose model that has already been trained on a smaller, targeted dataset and modifying its internal weights to become proficient in particular activities, terminology, or domains. By matching the model's output to domain specific specifications, this method lowers mistakes and improves handling of specialized material.

e done by Udita

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd

labels = ["Billing", "Technical Support", "Account Management", "Sales"]

ambiguous_messages = [
    "I need help with my account.",
    "My payment page is not working.",
    "I want to change my plan.",
    "Can someone help me update my information?",
    "I have a question before signing up."
]

def softmax_probs(model_used, text):
    prompt = f"""Classify the following customer message into one of these categories:
Billing, Technical Support, Account Management, Sales.

Message: "{text}"
Department:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model_used.device)

    decoder_input_ids = torch.tensor([[model_used.config.decoder_start_token_id]]).to(model_used.device)

    with torch.no_grad():
        outputs = model_used(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            decoder_input_ids=decoder_input_ids
        )

    logits = outputs.logits[:, -1, :]

    label_tokens = [tokenizer.encode(label, add_special_tokens=False)[0] for label in labels]
    label_logits = logits[:, label_tokens]

    probs = F.softmax(label_logits, dim=-1).cpu().numpy()[0]

    return dict(zip(labels, probs))

results = []

for msg in ambiguous_messages:

    base_pred = normalize(classify_zero_shot(msg))
    base_probs = softmax_probs(model, msg)

    tuned_pred = normalize(classify_finetuned(msg))
    tuned_probs = softmax_probs(tuned_model, msg)

    results.append({
        "Message": msg,
        "Base Prediction": base_pred,
        "Base Billing %": round(base_probs["Billing"]*100,2),
        "Base Tech %": round(base_probs["Technical Support"]*100,2),
        "Base Account %": round(base_probs["Account Management"]*100,2),
        "Base Sales %": round(base_probs["Sales"]*100,2),
        "Fine Prediction": tuned_pred,
        "Fine Billing %": round(tuned_probs["Billing"]*100,2),
        "Fine Tech %": round(tuned_probs["Technical Support"]*100,2),
        "Fine Account %": round(tuned_probs["Account Management"]*100,2),
        "Fine Sales %": round(tuned_probs["Sales"]*100,2),
    })

results_df = pd.DataFrame(results)
results_df

f and g done by Evan

1f)

The model learns our company's unique labels and work patterns by fine-tuning on modest internal data. Overfitting is the primary concern since a short dataset could lead to the model memorizing examples rather than effectively applying generalizations to newly created communications.

1g)

When the job is specific, performed frequently, and supported by data with labels, fine-tuning works better. When the task is straightforward, less expensive to complete, or the starting architecture already functions adequately, prompt engineering is preferable.